In [ ]:
from agent_framework import (
    Executor,
    FunctionExecutor,
    WorkflowContext,
    handler,
    executor,
)
import inspect
from typing import get_origin, Never

In [ ]:
# Note: class based executors can have multiple handlers
# Note: function based executors have a single handler, which is the function itself
# Note: class based executors require an id to be provided, function based executors use the function name as the id if not provided
# Note: handlers cannot have the same signature. If two handlers have union types overlap, handlers are registered in reverse order, so the most recently added handler that matches the input type will be used.
# Note: ^MAF does not have documentation for this behaviour, yet but I have tested to confirm it. Handlers are executed in the order they are returned from the executor's _handler_specs property, which is in reverse order of registration.

## Signature / Outputs Guide

`ctx: WorkflowContext[str]`: used with send_message to send a [str] message to the next executor

`ctx: WorkflowContext[Never, str]`: used with ctx.yield_output to send a [str] workflow output to the caller

`ctx: WorkflowContext[str, str]`: send_message[str] and yield_output[str]


Explicitly define type parameters:

`@handler(input=str | int, output=str, workflow_output=bool)`: 

Implicitly define type parameters (will be inferred through introspection)

```
@handler()
async def my_method(self, input: str | int, ctx: WorkflowContext[str, bool])
``` 



In [124]:
class UpperCase(Executor):
    def __init__(self, id: str = "upper_case_executor") -> None:
        super().__init__(id=id)
    @handler
    async def to_upper_case(self, text: str, ctx: WorkflowContext[str]) -> None:
        """Convert the input to uppercase and forward it to the next node."""
        await ctx.send_message(text.upper())

    # Note: this handle will always be called for a str type because handlers are registered in reverse order, so the most recently added handler that matches the input type will be used.
    @handler
    async def to_lower_case(self, text: str | int, ctx: WorkflowContext[str]) -> None:
        """Convert the input to lowercase and forward it to the next node."""
        await ctx.send_message(str(text)[:3].lower())

    @handler
    async def to_double_integer(self, number: int, ctx: WorkflowContext[int, str]) -> None:
        """Double the input integer and forward it to the next node."""
        await ctx.send_message(number * 2)
        await ctx.yield_output("done")

In [111]:
@executor()
async def reverse_text(text: str, ctx: WorkflowContext[str, int]) -> None:
    """Reverse the input text and forward it to the next node."""
    await ctx.send_message(text[::-1])

# Note: the explicit types are not checked 
# by the framework. If you provide incorrect types, it may lead to runtime errors or unexpected behavior.
@executor(input=str, output=str, workflow_output=float)
async def reverse_text_invalid_types(text, ctx: WorkflowContext[Never, int]) -> None:
    """Reverse the input text and forward it to the next node."""
    await ctx.send_message(text[::-1])

# Note: @executor expects your function to accept 1 or 2 params (message) or (message, context) it
# blindly treats params[0] as the message regardless of name or annotation. This means if your
# first parameter is context, it will assume that is actually your message. This is a silent failure that should be avoided.
@executor()
async def reverse_text_invalid_parameters(ctx: WorkflowContext[str, int]) -> None:
    """Reverse the input text and forward it to the next node."""
    await ctx.send_message("some message")

In [99]:
from typing import TypedDict


class HandlerSpecInfo(TypedDict):
    handler_name: str
    input_type: list[type]
    output_types: list[type]
    workflow_output_types: list[type]
    doc_string: str | None
    input_parameter_names: list[str]

In [114]:
def get_handler_doc_string(executor: Executor, input_type: type) -> str | None:
    return executor._handlers.get(input_type).__doc__

# Note: Executor prevents handlers from having more than one message_type
def get_handler_parameter_names(executor: Executor, input_type: type) -> list[str]:
    print(input_type)
    handler = executor._handlers.get(input_type)
    if handler is None:
        raise ValueError(f"No handler found for input type {input_type}")
    parameters = inspect.signature(handler).parameters
    parameter_names = [parameters[p].name for p in parameters if not get_origin(parameters[p].annotation) is WorkflowContext] # remove the Workflow context as it is not user provided input
    return parameter_names

In [113]:
def get_function_executor_metadata(executor: FunctionExecutor) -> list[HandlerSpecInfo]:
    if not isinstance(executor, FunctionExecutor):
        raise ValueError("Input must be an instance of Executor")
    handler_info: list[HandlerSpecInfo] = []
    # Note: Use executor._original_func to access the original function
    for handler_spec in executor._handler_specs:
        name = handler_spec["name"]
        input_type = handler_spec["message_type"]
        output_types = handler_spec["output_types"]
        workflow_output_types = handler_spec["workflow_output_types"]

        handler_info.append({
            "handler_name": name,
            "input_type": input_type,
            "output_types": output_types,
            "workflow_output_types": workflow_output_types,
            "doc_string": get_handler_doc_string(executor, input_type), # Note: handlers require unique input types
            "input_parameter_names": get_handler_parameter_names(executor, input_type),
        })

    return handler_info

In [ ]:
# Note: this works when the class is already instantiated
def get_instance_class_executor_metadata(executor: Executor) -> list[HandlerSpecInfo]:
    handler_info: list[HandlerSpecInfo] = []
    # Note: Use executor._original_func to access the original function
    for handler_spec in executor._handler_specs:
        name = handler_spec["name"]
        input_type = handler_spec["message_type"]
        output_types = handler_spec["output_types"]
        workflow_output_types = handler_spec["workflow_output_types"]

        handler_info.append({
            "handler_name": name,
            "input_type": input_type,
            "output_types": output_types,
            "workflow_output_types": workflow_output_types,
            "doc_string": get_handler_doc_string(executor, input_type), # Note: handlers require unique input types
            "input_parameter_names": get_handler_parameter_names(executor, input_type),
        })

    return handler_info

# Note: the parameter is type[Executor] (the class itself, not an instance).
# The @handler decorator stores a _handler_spec dict directly on each decorated method,
# so we can discover handlers from the class without instantiating it.
def get_class_executor_metadata(executor_class: type[Executor]) -> list[HandlerSpecInfo]:
    handler_info: list[HandlerSpecInfo] = []
    for attr_name in dir(executor_class):
        try:
            attr = getattr(executor_class, attr_name)
        except AttributeError:
            continue
        if callable(attr) and hasattr(attr, "_handler_spec"):
            # Note: getattr used instead of attr._handler_spec to satisfy the type checker.
            handler_spec = getattr(attr, "_handler_spec")
            parameters = inspect.signature(attr).parameters
            parameter_names = [
                parameters[p].name for p in parameters
                if parameters[p].name != "self"
                and not get_origin(parameters[p].annotation) is WorkflowContext
            ]
            handler_info.append({
                "handler_name": handler_spec["name"],
                "input_type": handler_spec["message_type"],
                "output_types": handler_spec["output_types"],
                "workflow_output_types": handler_spec["workflow_output_types"],
                "doc_string": attr.__doc__,
                "input_parameter_names": parameter_names,
            })
    return handler_info

In [125]:
get_class_executor_metadata(UpperCase)

[{'handler_name': 'to_double_integer',
  'input_type': int,
  'output_types': [int],
  'workflow_output_types': [str],
  'doc_string': 'Double the input integer and forward it to the next node.',
  'input_parameter_names': ['number']},
 {'handler_name': 'to_lower_case',
  'input_type': str | int,
  'output_types': [str],
  'workflow_output_types': [],
  'doc_string': 'Convert the input to lowercase and forward it to the next node.',
  'input_parameter_names': ['text']},
 {'handler_name': 'to_upper_case',
  'input_type': str,
  'output_types': [str],
  'workflow_output_types': [],
  'doc_string': 'Convert the input to uppercase and forward it to the next node.',
  'input_parameter_names': ['text']}]